In [1]:
import itertools
import logging
from pathlib import Path
from typing import Any, Callable, Sequence

import matplotlib.pyplot as plt
import mlflow
import nico2_lib as n2l
import numpy as np
import pandas as pd
import polars as pl
import scanpy as sc
import seaborn as sns
from experiment import Celltype, Dataset, Model, NumericArray, Result, Sample
from numpy import number
from sqlalchemy import event
from sqlalchemy.engine import Engine
from sqlmodel import Session, create_engine, select
import scipy
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

/Users/egerc/Documents/Projects/notebook_repository/notebooks/2026-04-24T06-54-05Z/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



In [2]:
exp_name = "Default"
benchmark_experiment = mlflow.get_experiment_by_name(exp_name)
if not benchmark_experiment:
    raise ValueError(f"Experiment '{exp_name}' not found.")
runs = mlflow.search_runs(experiment_ids=[benchmark_experiment.experiment_id])
if runs.empty:
    raise RuntimeError(f"No runs found in experiment '{exp_name}'.")
last_run_id = runs.sort_values("start_time", ascending=False)["run_id"].iloc[0]
logger.info(f"Using run_id: {last_run_id}")
logger.info("Downloading 'database.db'...")
database_path = mlflow.artifacts.download_artifacts(
    run_id=last_run_id, artifact_path="database.db"
)
logger.info(f"Local path: {database_path}")
sqlite_url = f"sqlite:///{database_path}"
engine = create_engine(sqlite_url, echo=True)
logger.info("Database engine initialized.")

INFO: Using run_id: 9ba2213ff31744f18e8ab6940549b0bd
INFO: Downloading 'database.db'...
INFO: Local path: /var/folders/qm/v_v5_1r52bx792m7x2mh177c0000gn/T/tmphq2h85g8/database.db
INFO: Database engine initialized.


In [3]:
database_path = "/Users/egerc/Documents/Projects/notebook_repository/notebooks/2026-03-11T09-44-10Z/database.db"
logger.info(f"Local path: {database_path}")
sqlite_url = f"sqlite:///{database_path}"
engine = create_engine(sqlite_url, echo=True)
logger.info("Database engine initialized.")

INFO: Local path: /Users/egerc/Documents/Projects/notebook_repository/notebooks/2026-03-11T09-44-10Z/database.db
INFO: Database engine initialized.


In [34]:
def placeholder(result: Result) -> tuple[float, float, float, float]:
    return 0.0, 1.0, 2.0, 3.0

In [ ]:
def _create_avg_pairwise_metric_fn(
    metric_fn: Callable[[NumericArray, NumericArray], float],
) -> Callable[[NumericArray], float]:
    def pairwise_metric_fn(
        features: NumericArray,
    ) -> float:
        eps = 1e-8
        return np.array(
            [
                metric_fn(a + eps, b + eps)
                for a, b in itertools.combinations(features.T, 2)
            ]
        ).mean()

    return pairwise_metric_fn


def create_embedding_evaluator(
    correlation_func: Callable[[NumericArray, NumericArray], float],
) -> Callable[[Result], tuple[float, float, float, float]]:
    average_pairwise_correlation_func = _create_avg_pairwise_metric_fn(
        metric_fn=correlation_func
    )

    def pairwise_correlation_func(
        result: Result,
    ) -> tuple[float, float, float, float]:
        results: tuple[float, float, float, float] = tuple(
            [
                average_pairwise_correlation_func(embedding)
                for embedding in [
                    result.global_model_embedding_reference,
                    result.global_model_embedding_query,
                    result.celltype_model_embedding_reference,
                    result.celltype_model_embedding_query,
                ]
            ]
        )
        return results

    return pairwise_correlation_func

In [ ]:
METRIC_FNS: dict[
    str, dict[str, Callable[[Result], tuple[float, float, float, float]]]
] = {
    "placeholder": {
        "placeholder": placeholder,
    },
    "embedding_autocorrelation": {
        "pearsonr": create_embedding_evaluator(
            n2l.mt.pearson_metric,
        ),
        "spearmanr": create_embedding_evaluator(
            n2l.mt.spearman_metric,
        ),
        "cosine_similarity": create_embedding_evaluator(
            n2l.mt.cosine_similarity_metric,
        ),
    },
}

In [47]:
from itertools import product


with Session(engine) as session:
    results = session.exec(select(Result)).all()
    rows: list[dict[str, Any]] = []
    for result in results:
        for metric_category, function_mapping in METRIC_FNS.items():
            for function_name, metric_function in function_mapping.items():
                metrics = metric_function(result)
                for value, (model_scope, dataset_split) in zip(
                    metrics, product(["global", "celltype"], ["reference", "query"])
                ):
                    rows.append(
                        {
                            "dataset_name": result.celltype.dataset.name,
                            "celltype": result.celltype.name,
                            "sample_id": result.sample.id_of_sample,
                            "model_name": result.model.name,
                            "metric_category": metric_category,
                            "function_name": function_name,
                            "model_scope": model_scope,
                            "dataset_split": dataset_split,
                            "value": value,
                        }
                    )
    results_df = pd.DataFrame(rows)

2026-04-29 16:03:27,904 INFO sqlalchemy.engine.Engine BEGIN (implicit)


2026/04/29 16:03:27 INFO sqlalchemy.engine.Engine: BEGIN (implicit)


2026-04-29 16:03:27,907 INFO sqlalchemy.engine.Engine SELECT result.id, result.global_model_embedding_reference, result.global_model_counts_reference, result.celltype_model_embedding_reference, result.celltype_model_counts_reference, result.global_model_embedding_query, result.global_model_counts_query, result.celltype_model_embedding_query, result.celltype_model_counts_query, result.celltype_id, result.model_id, result.sample_id 
FROM result


2026/04/29 16:03:27 INFO sqlalchemy.engine.Engine: SELECT result.id, result.global_model_embedding_reference, result.global_model_counts_reference, result.celltype_model_embedding_reference, result.celltype_model_counts_reference, result.global_model_embedding_query, result.global_model_counts_query, result.celltype_model_embedding_query, result.celltype_model_counts_query, result.celltype_id, result.model_id, result.sample_id 
FROM result


2026-04-29 16:03:27,909 INFO sqlalchemy.engine.Engine [cached since 4811s ago] ()


...

2026/04/29 16:03:30 INFO sqlalchemy.engine.Engine: SELECT celltype.id AS celltype_id, celltype.name AS celltype_name, celltype.reference_counts_matrix AS celltype_reference_counts_matrix, celltype.reference_pca_embedding AS celltype_reference_pca_embedding, celltype.reference_umap_embedding AS celltype_reference_umap_embedding, celltype.reference_adjacency_matrix AS celltype_reference_adjacency_matrix, celltype.query_counts_matrix AS celltype_query_counts_matrix, celltype.query_pca_embedding AS celltype_query_pca_embedding, celltype.query_umap_embedding AS celltype_query_umap_embedding, celltype.query_adjacency_matrix AS celltype_query_adjacency_matrix, celltype.n_pcs AS celltype_n_pcs, celltype.n_neighbours AS celltype_n_neighbours, celltype.dataset_id AS celltype_dataset_id 
FROM celltype 
WHERE celltype.id = ?


2026-04-29 16:03:30,861 INFO sqlalchemy.engine.Engine [cached since 4396s ago] (38,)


2026/04/29 16:03:30 INFO sqlalchemy.engine.Engine: [cached since 4396s ago] (38,)


2026-04-29 16:03:30,939 INFO sqlalchemy.engine.Engine ROLLBACK


2026/04/29 16:03:30 INFO sqlalchemy.engine.Engine: ROLLBACK


In [48]:
results_df

,dataset_name,celltype,sample_id,model_name,metric_category,function_name,model_scope,dataset_split,value
0,mouse_intestine_spatial,BZE,0,nmf_3_raw,placeholder,placeholder,global,reference,0.000000
1,mouse_intestine_spatial,BZE,0,nmf_3_raw,placeholder,placeholder,global,query,1.000000
2,mouse_intestine_spatial,BZE,0,nmf_3_raw,placeholder,placeholder,celltype,reference,2.000000
3,mouse_intestine_spatial,BZE,0,nmf_3_raw,placeholder,placeholder,celltype,query,3.000000
4,mouse_intestine_spatial,BZE,0,nmf_3_raw,embedding_autocorrelation,pearsonr,global,reference,0.079251
...,...,...,...,...,...,...,...,...,...
9115,mouse_intestine_pseudospatial,pDC,4,scvi_3_raw,embedding_autocorrelation,spearmanr,celltype,query,-0.327273
9116,mouse_intestine_pseudospatial,pDC,4,scvi_3_raw,embedding_autocorrelation,cosine_similarity,global,reference,-0.325171
9117,mouse_intestine_pseudospatial,pDC,4,scvi_3_raw,embedding_autocorrelation,cosine_similarity,global,query,0.815359
9118,mouse_intestine_pseudospatial,pDC,4,scvi_3_raw,embedding_autocorrelation,cosine_similarity,celltype,reference,-0.329024
